<center>
  <h3>More than a Hack 2027 &middot; Phase 1</h3>
  <h2>Frame-level Ball Possession &mdash; 013 Model Terawasi dari Pertandingan Lain</h2>

  <p><strong>Metrica Sports &times; FC Barcelona Innovation Hub &times; Mobile World Capital</strong></p>
</center>

---

### Three Makasih

- **Anggota 1** (<span style="color:gray">email</span>)
- **Anggota 2** (<span style="color:gray">email</span>)
- **Anggota 3** (<span style="color:gray">email</span>)

---

## Ringkasan

Riset literatur (`experiments/RESEARCH.md`) menunjukkan bahwa model yang **dilatih terawasi** mengalahkan aturan tangan untuk bola mati secara telak: TCN pada data tracking mencapai 89,2% akurasi hidup-mati dan 86,2% akurasi kepemilikan, masing-masing 33,2 dan 7,3 poin di atas metode berbasis aturan, tetapi dengan 35 pertandingan berlabel. Kita punya tiga pertandingan Metrica (Sample Game 1, 2, dan 3) dengan label turunan peristiwa. Eksperimen ini melatih satu TCN dengan dua kepala (bola mati dan tim penguasa) pada pertandingan-pertandingan itu, memvalidasinya **antar pertandingan** (latih dua game, uji game ketiga), lalu memindahkannya ke pertandingan ini.

Fitur memadukan temuan literatur: pemicu konfigurasi pemain berdasarkan hukum IFAB (jarak ke tanda sudut dan titik tengah, pemain di luar garis samping, pemain di area gawang, pemain di babak sendiri), gerak kolektif (kecepatan dan proporsi pemain bergerak), bentuk tim terhadap arah serang, serta saluran bola (kendali lunak, kecepatan bola, jarak ke garis, jarak pemain terdekat) yang **disembunyikan** pada potongan acak saat pelatihan agar model bisa bekerja ketika bola hilang.

Tidak ada label tangan pada pertandingan ini dan ground truth tidak dipakai untuk melatih atau memilih apa pun.


# Metodologi

**peristiwa Metrica Game 1, 2, dan 3 → label H/A/D per frame → fitur dataset-agnostik (konfigurasi pemain + gerak + bentuk tim + saluran bola) → TCN dua kepala dengan masking bola → validasi silang antar pertandingan (latih dua game, uji game ketiga, tiga lipatan) vs gradient boosting per langkah → model akhir dilatih pada ketiga game → diterapkan ke pertandingan ini → p(bola mati) dan p(H) per frame → diagnostik internal.**

| asumsi | nilai | alasan |
|---|---|---|
| data latih | Metrica Sample Game 1, 2, dan 3 (Game 3 sebelumnya salah dibaca: periodenya tidak diturunkan dari peristiwa, sekarang diperbaiki) | data pertandingan lain, penyedia sama dengan tantangan; harus dikreditkan; Game 3 berformat EPTS seperti ATD pertandingan ini |
| label | turunan algoritmik dari peristiwa (lihat `other_games.py`) | kasar: porsi D 24 sampai 36% |
| laju sampel, konteks | 5 fps, TCN dilatasi 1 sampai 16 (sekitar 25 detik) | sama dengan 008 |
| masking bola | 50% potongan bola disembunyikan seluruhnya, sisanya potongan acak | meniru gap bola panjang |
| kepala | bola mati (BCE pada semua frame berlabel) dan tim (BCE pada frame hidup) | dua keputusan terpisah seperti pada literatur TCN |

# Environment

Dijalankan dari dalam repositori, dilatih di GPU lokal. Data Metrica diunduh ke cache gitignored `experiments/_cache/ext_metrica/` dan harus dikreditkan di dokumen teknis (https://github.com/metrica-sports/sample-data). Tidak ada data pertandingan ini yang dikirim ke mana pun.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
for p in (ROOT, ROOT / "starter", ROOT / "evaluation", ROOT / "experiments"):
    sys.path.insert(0, str(p))

import subprocess, time
import warnings
warnings.filterwarnings("ignore")
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

import common
import features as ft
import other_games as og
from games import GAME, GAMES, anchor_video_frame, half_video_blocks, source_frames

C1, C2, C3, INK, GREY = "#e94f37", "#393e41", "#f6f7eb", "#393e41", "#a3a7a6"
mpl.rcParams.update({
    "figure.facecolor": C3, "savefig.facecolor": C3, "axes.facecolor": "white",
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "legend.frameon": False,
    "figure.dpi": 110, "axes.prop_cycle": mpl.cycler(color=[C1, C2, GREY]),
})
def panel(ax, l): ax.text(-0.09, 1.07, l, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")

FPS = 25; STRIDE = 5
N = source_frames("tactical")
BLOCKS3 = half_video_blocks("BAR-ATH")
GRADED = np.zeros(N, bool); HALF = np.zeros(N, int)
for h, a, b in BLOCKS3: GRADED[a:b + 1] = True; HALF[a:b + 1] = h
GRADED_MIN = GRADED.sum() / FPS / 60
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"perangkat: {DEV}" + (f" ({torch.cuda.get_device_name(0)})" if DEV == "cuda" else ""))

# Approach: TCN Terawasi, Dilatih pada Pertandingan Lain

## [1] Data pelatihan dan fitur

In [ ]:
GAMES_TRAIN = {}
for n in (1, 2, 3):
    g = og.load(n); half = g["half"]
    F = ft.build(g["P"], g["B"], g["T"], half)
    ix = np.flatnonzero(half > 0)[::STRIDE]
    GAMES_TRAIN[n] = dict(player=F["player"][:, ix], ball=F["ball"][:, ix], valid=F["pvalid"][:, ix], y=g["y"][ix].astype(int), half=half[ix], ix=ix)
    y = GAMES_TRAIN[n]["y"]
    print(f"game {n}: {len(ix)} langkah (5 fps) | berlabel {np.mean(y >= 0):.1%} | mati {np.mean(y[y >= 0] == 2):.1%} | bola terlihat {F['obs'][ix].mean():.1%}")

P, B, T = common.load_atd_arrays()
FO = ft.build(P[..., :2], B[:, :2], T, HALF)
IXO = {h: np.arange(a, b + 1, STRIDE) for h, a, b in BLOCKS3}
print(f"pertandingan ini: bola terlihat pada frame dinilai {FO['obs'][GRADED].mean():.1%}")

mu = np.nanmean(np.concatenate([G["player"][0] for G in GAMES_TRAIN.values()]), 0)
sd = np.nanstd(np.concatenate([G["player"][0] for G in GAMES_TRAIN.values()]), 0) + 1e-6
print("fitur pemain:", len(ft.PLAYER_FEATS), "| saluran bola:", len(ft.BALL_FEATS))

## [2] Gambaran sinyal

Rata-rata fitur pemain pada frame bola mati dan hidup di game latih, dan rata-ratanya pada semua frame dinilai di pertandingan ini. Pada pertandingan ini tidak ada label, jadi hanya distribusi fitur yang dibandingkan, bukan label.

In [ ]:
rows = []
for i, name in enumerate(ft.PLAYER_FEATS):
    d, a = [], []
    for G in GAMES_TRAIN.values():
        v = G["player"][0][:, i]; y = G["y"]
        d.append(np.nanmean(v[y == 2])); a.append(np.nanmean(v[(y >= 0) & (y < 2)]))
    ours = np.nanmean(FO["player"][0][GRADED, i])
    rows.append({"fitur": name, "mati (Metrica)": round(np.mean(d), 2), "hidup (Metrica)": round(np.mean(a), 2), "pertandingan ini (semua frame dinilai)": round(float(ours), 2)})
display(pd.DataFrame(rows))

Fitur yang paling memisahkan mati dari hidup adalah gerak kolektif (kecepatan semua pemain dan proporsi pemain yang bergerak). Rata-rata di pertandingan ini berada di antara dua kelompok, wajar karena ia campuran mati dan hidup, dan menunjukkan bahwa kedua dataset berada di domain yang sebanding.

## [3] Model dan pelatihan

TCN dua arah dengan blok residual berdilatasi, dua kepala keluaran. Pelatihan memakai potongan acak 384 langkah (sekitar 77 detik) dari game latih, dengan dua sudut pandang (tim *a* = H atau A) dan masking bola.

In [ ]:
NCH = len(ft.PLAYER_FEATS) + 1 + len(ft.BALL_FEATS)

class TCN2(nn.Module):
    def __init__(self, cin=NCH, ch=64, dil=(1, 2, 4, 8, 16, 1), k=5, p=0.15):
        super().__init__()
        self.inp = nn.Conv1d(cin, ch, 1)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Conv1d(ch, ch, k, padding=d * (k // 2), dilation=d), nn.GELU(), nn.Dropout(p)) for d in dil])
        self.norm = nn.ModuleList([nn.GroupNorm(8, ch) for _ in dil])
        self.head = nn.Conv1d(ch, 2, 1)                       # kanal 0: bola mati, kanal 1: tim a menguasai
    def forward(self, x):
        h = self.inp(x)
        for blk, nm in zip(self.blocks, self.norm): h = h + blk(nm(h))
        return self.head(h)

def inputs(pl, bl, valid, mask_ball):
    f = np.nan_to_num((pl - mu) / sd).astype(np.float32)
    b = np.nan_to_num(np.where(mask_ball[:, None], 0.0, bl)).astype(np.float32)   # NaN (e.g. rolled speed without samples) -> 0
    return np.concatenate([f, valid[:, None].astype(np.float32), b], 1)

def targets(y, view):
    dead = (y == 2).astype(np.float32); team = ((y == view)).astype(np.float32)
    return dead, team, (y >= 0).astype(np.float32), ((y == 0) | (y == 1)).astype(np.float32)

def random_mask(T_, rng):
    if rng.random() < 0.5: return np.ones(T_, bool)
    m = np.zeros(T_, bool); i = 0
    while i < T_:
        Ln = int(rng.integers(8, 160))
        if rng.random() < 0.5: m[i:i + Ln] = True
        i += Ln
    return m

def sequences(games):
    out = []
    for G in games.values():
        for h in (1, 2):
            s = np.flatnonzero(G["half"] == h)
            if len(s) > 400: out.append((G, s))
    return out

def train(games, seed, steps=1500, bs=32, crop=384):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    seqs = sequences(games)
    net = TCN2().to(DEV); opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps)
    for it in range(steps):
        X, Yd, Yt, Md, Mt = [], [], [], [], []
        for _ in range(bs):
            G, s = seqs[int(rng.integers(len(seqs)))]; st = int(rng.integers(0, len(s) - crop)); sl = s[st:st + crop]; v = int(rng.integers(0, 2))
            X.append(inputs(G["player"][v][sl], G["ball"][v][sl], G["valid"][v][sl], random_mask(crop, rng)))
            d, t, md, mt = targets(G["y"][sl], v); Yd.append(d); Yt.append(t); Md.append(md); Mt.append(mt)
        X = torch.tensor(np.stack(X)).permute(0, 2, 1).to(DEV)
        Yd, Yt, Md, Mt = [torch.tensor(np.stack(a)).to(DEV) for a in (Yd, Yt, Md, Mt)]
        out = net(X); bce = nn.functional.binary_cross_entropy_with_logits
        loss = (bce(out[:, 0], Yd, reduction="none") * Md).sum() / Md.sum().clamp(min=1) + (bce(out[:, 1], Yt, reduction="none") * Mt).sum() / Mt.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net.eval()

@torch.no_grad()
def predict(net, pl, bl, valid, mask_all):
    outs = []
    for v in (0, 1):
        x = torch.tensor(inputs(pl[v], bl[v], valid[v], np.full(len(pl[v]), mask_all))).T[None].to(DEV)
        outs.append(torch.sigmoid(net(x))[0].cpu().numpy())
    return 0.5 * (outs[0][0] + outs[1][0]), 0.5 * (outs[0][1] + 1 - outs[1][1])

t0 = time.time(); _ = train(GAMES_TRAIN, 0, steps=40); print(f"40 langkah latih: {time.time() - t0:.1f} dtk")

## [4] Validasi silang antar pertandingan

Latih pada satu game, uji pada game lain, hanya pada langkah berlabel. Dua kondisi bola: **disembunyikan seluruhnya** (kondisi frame tanpa bola) dan **tersedia**. Pembanding: gradient boosting per langkah dengan fitur yang sama (tanpa konteks temporal luas). Akurasi tim dihitung pada langkah hidup saja.

In [ ]:
def gbdt_fit(games, mask_ball):
    X, yd, yt, mt = [], [], [], []
    for G in games.values():
        for v in (0, 1):
            m = G["y"] >= 0
            f = inputs(G["player"][v], G["ball"][v], G["valid"][v], np.full(len(G["y"]), mask_ball))[m]
            d, t, _, mtm = targets(G["y"][m], v); X.append(f); yd.append(d); yt.append(t); mt.append(mtm.astype(bool))
    X, yd, yt, mt = map(np.concatenate, (X, yd, yt, mt))
    gd = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=200, random_state=0).fit(X, yd)
    gt = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=200, random_state=0).fit(X[mt], yt[mt])
    return gd, gt

def gbdt_predict(models, G, mask_ball):
    gd, gt = models; pd_, pt = [], []
    for v in (0, 1):
        f = inputs(G["player"][v], G["ball"][v], G["valid"][v], np.full(len(G["y"]), mask_ball))
        pd_.append(gd.predict_proba(f)[:, 1]); pt.append(gt.predict_proba(f)[:, 1])
    return 0.5 * (pd_[0] + pd_[1]), 0.5 * (pt[0] + 1 - pt[1])

rows = []
for te in (1, 2, 3):
    tr = [k for k in GAMES_TRAIN if k != te]; TR = {k: GAMES_TRAIN[k] for k in tr}
    G = GAMES_TRAIN[te]; y = G["y"]; lab = y >= 0; alive = (y == 0) | (y == 1)
    nets = [train(TR, s) for s in range(3)]
    for mask_ball, cond in ((True, "bola disembunyikan"), (False, "bola tersedia")):
        preds = [predict(n, G["player"], G["ball"], G["valid"], mask_ball) for n in nets]
        pdead = np.mean([p[0] for p in preds], 0); pteam = np.mean([p[1] for p in preds], 0)
        gb = gbdt_predict(gbdt_fit(TR, mask_ball), G, mask_ball)
        for name, (pdn, ptn) in (("TCN", (pdead, pteam)), ("gradient boosting", gb)):
            rows.append({"latih": "+".join(map(str, tr)), "uji": f"game {te}", "kondisi": cond, "model": name,
                         "AUC mati": roc_auc_score(y[lab] == 2, pdn[lab]), "akurasi mati": accuracy_score(y[lab] == 2, pdn[lab] > 0.5),
                         "akurasi tim (hidup)": accuracy_score(y[alive] == 0, ptn[alive] > 0.5)})
CV = pd.DataFrame(rows); display(CV.round(3))

Tabel dibaca dalam tiga arah: (a) apakah TCN mengalahkan gradient boosting per langkah pada kondisi bola disembunyikan, (b) seberapa jauh bola yang tersedia menaikkan akurasi, (c) konsistensi di kedua arah game. Karena label turunan peristiwa kasar, akurasi absolut bukan klaim tentang ground truth tantangan, hanya kesesuaian dengan peristiwa Metrica.

## [5] Model akhir dan penerapan ke pertandingan ini

Ensembel tiga seed dilatih pada ketiga game, lalu diterapkan ke pertandingan ini pada dua kondisi: **bola nyata** (dengan hilangnya bola apa adanya) untuk p(bola mati), dan **bola disembunyikan** untuk p(tim) agar tidak menggandakan bukti bola di decoder.

In [ ]:
finals = [train(GAMES_TRAIN, s) for s in range(3)]
PD_REAL = np.full(N, np.nan); PD_MASK = np.full(N, np.nan); PT_MASK = np.full(N, 0.5)
for h, a, b in BLOCKS3:
    ix = IXO[h]
    pl, bl, vd = FO["player"][:, ix], FO["ball"][:, ix], FO["pvalid"][:, ix]
    real = np.mean([predict(n, pl, bl, vd, False)[0] for n in finals], 0)
    mask = [predict(n, pl, bl, vd, True) for n in finals]
    full = np.arange(a, b + 1)
    PD_REAL[a:b + 1] = np.interp(full, ix, real)
    PD_MASK[a:b + 1] = np.interp(full, ix, np.mean([m[0] for m in mask], 0))
    PT_MASK[a:b + 1] = np.interp(full, ix, np.mean([m[1] for m in mask], 0))
np.savez(Path(common.CACHE) / "p013.npz", pd_real=PD_REAL, pd_mask=PD_MASK, pt_mask=PT_MASK)

def runs_stats(mask):
    s, e = common.runs(mask); d = (e - s) / FPS
    return len(s), (float(np.median(d)) if len(d) else float("nan"))
for name, p in (("p(mati), bola nyata", PD_REAL), ("p(mati), bola disembunyikan", PD_MASK)):
    m = GRADED & (p > 0.5); n, med = runs_stats(m)
    print(f"{name}: p>0,5 pada {m.sum() / GRADED.sum():.1%} frame dinilai | {n} henti permainan, median {med:.1f} dtk | rata-rata p {np.nanmean(p[GRADED]):.3f}")
print("acuan Metrica (label peristiwa): porsi mati 24 sampai 36%, median henti permainan 13 sampai 21 dtk")

fig, ax = plt.subplots(2, 1, figsize=(12, 3.6))
for a_, (h, s, e) in zip(ax, BLOCKS3):
    t = np.arange(s, e + 1)[::25] / FPS / 60
    a_.plot(t, PD_REAL[s:e + 1:25], color=C1, lw=0.7, label="bola nyata"); a_.plot(t, PD_MASK[s:e + 1:25], color=GREY, lw=0.6, label="bola disembunyikan")
    a_.axhline(0.5, color=INK, lw=0.6, ls="--"); a_.set(ylim=(0, 1), ylabel="p(mati)", title=f"Babak {h}: p(bola mati)")
ax[1].set_xlabel("menit video"); ax[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.4), ncol=2)
panel(ax[0], "A"); panel(ax[1], "B"); fig.tight_layout(); plt.show()

# Ringkasan 013

__SUMMARY__